# RQ1 — Sample-based reproduction of the classifier

Redoes the LLM classification on samples, with the same prompt used in the study, and compares the
result with the study's data. The new classifications are generated by
`scripts/reproduce_classification_sample.py` (see `data/rq01/README.md`) and are stored in
`results/rq01/sample_reproduction/`:

1. **random sample** of 30 files, compared with runs 1 to 3;
2. **validation**: the 97 files of the manual annotation, to recompute κ against the consensus.

The output of an LLM is not deterministic, so equality is not expected. The reference is the
agreement among the study's own runs (83.5% on average, notebook `01`): if the new classification
agrees with the study at a similar level, the procedure has been reproduced.

In [1]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data" / "collection").exists():
    ROOT = ROOT.parent

CLASSIFIER = ROOT / "data/rq01/classifier"
NEW = ROOT / "results/rq01/sample_reproduction/rq01_classification_run_sample.csv"
OUT = ROOT / "results/rq01/sample_reproduction"

CATEGORIES = [
    "run_and_build", "project_overview", "doc_metadata", "test_instructions",
    "project_structure", "system_design", "contribution_guidelines", "code_conventions",
    "ai_rules", "performance_&_resource_management", "troubleshooting", "security",
    "deployment_&_infrastructure",
]


def load(path):
    d = pd.read_csv(path)
    d["key"] = d["repo"].str.strip().str.lower() + "|" + d["arquivo_analisado"].str.strip()
    for c in CATEGORIES:
        d[c] = d[c].fillna(0).astype(int)
    return d.set_index("key")


new = load(NEW)
keys = new.index
print(f"Files in the new classification: {len(new)}")
runs = {"new": new.loc[keys, CATEGORIES]}
for i in (1, 2, 3):
    runs[f"run {i}"] = load(CLASSIFIER / f"rq01_classification_run_{i}.csv").loc[keys, CATEGORIES]

Files in the new classification: 30


## 1. Random sample: agreement per pair of runs

Percentage of equal decisions (file × category) on the files of the sample.

In [2]:
pairs = pd.DataFrame([{"pair": f"{a} × {b}", "agreement_pct": 100 * (runs[a] == runs[b]).values.mean()}
                      for a, b in combinations(runs, 2)])
pairs.to_csv(OUT / "agreement_pairs.csv", index=False)
pairs.round(1)

,pair,agreement_pct
0,new × run 1,82.6
1,new × run 2,92.1
2,new × run 3,84.6
3,run 1 × run 2,82.8
4,run 1 × run 3,82.6
5,run 2 × run 3,84.4


## 1. Random sample: agreement per category (new × run 1)

In [3]:
per_category = pd.DataFrame({
    "new_x_run_1": 100 * (runs["new"] == runs["run 1"]).mean(),
    "mean_across_study_runs": 100 * pd.concat(
        [(runs[a] == runs[b]).mean() for a, b in combinations(["run 1", "run 2", "run 3"], 2)],
        axis=1).mean(axis=1),
})
per_category.to_csv(OUT / "agreement_per_category.csv")
per_category.round(1)

,new_x_run_1,mean_across_study_runs
run_and_build,93.3,91.1
project_overview,93.3,95.6
doc_metadata,76.7,73.3
test_instructions,90.0,84.4
project_structure,80.0,86.7
system_design,66.7,75.6
contribution_guidelines,73.3,73.3
code_conventions,90.0,86.7
ai_rules,80.0,71.1
performance_&_resource_management,93.3,91.1


## 1. Random sample: reading

In [4]:
new_vs_study = pairs[pairs["pair"].str.startswith("new")]["agreement_pct"].mean()
within_study = pairs[~pairs["pair"].str.startswith("new")]["agreement_pct"].mean()
print(f"Mean agreement of the new classification with the study's runs: {new_vs_study:.1f}%")
print(f"Mean agreement among the study's runs (same files): {within_study:.1f}%")

Mean agreement of the new classification with the study's runs: 86.4%
Mean agreement among the study's runs (same files): 83.2%


## 2. Validation: κ against the annotators' consensus

Same computation as in notebook `01` (Cohen's κ per category, mean over the 13, on the 91 files with
consensus), now with the new classification of the 97 files of the manual annotation.

In [5]:
VALIDATION = ROOT / "results/rq01/sample_reproduction/rq01_classification_run_validation.csv"
LABELS = ROOT / "data/rq01/manual_annotation/rq01_manual_labels.csv"


def kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    po = (a == b).mean()
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return float("nan") if pe == 1 else (po - pe) / (1 - pe)


labels = pd.read_csv(LABELS, keep_default_na=False)
labels = labels[labels["consenso"].str.strip() != ""].reset_index(drop=True)
sets = labels["consenso"].apply(lambda v: {c.strip() for c in v.split(",") if c.strip()})
consensus = pd.DataFrame({c: sets.apply(lambda x: int(c in x)) for c in CATEGORIES})
val_keys = (labels["repo"].str.lower() + "|" + labels["arquivo"]).values

comparisons = {"new validation": VALIDATION, "study validation": CLASSIFIER / "rq01_validation_run.csv"}
k = pd.DataFrame({name: pd.Series({c: kappa(consensus[c], load(path).loc[val_keys, c].values) for c in CATEGORIES})
                  for name, path in comparisons.items() if path.exists()})
k.loc["mean"] = k.mean()
k.to_csv(OUT / "reproduced_validation_kappa.csv")
k.round(3)

,new validation,study validation
run_and_build,0.721,0.802
project_overview,0.806,0.758
doc_metadata,0.629,0.716
test_instructions,0.961,0.961
project_structure,0.901,0.849
system_design,0.768,0.768
contribution_guidelines,0.799,0.729
code_conventions,0.795,0.785
ai_rules,0.792,0.607
performance_&_resource_management,0.716,0.834
